### 1.3.7.5. Matrix Chain Rule

$$
L = \tfrac{1}{2}\,\|W\mathbf{x} - \mathbf{b}\|^{2}
\;\Longrightarrow\;
\frac{\partial L}{\partial W} = (W\mathbf{x} - \mathbf{b})\,\mathbf{x}^{\top} .
$$

**Explanation:**

The matrix chain rule propagates a derivative through compositions where the intermediate quantities are vectors or matrices, generalizing the [multivariable chain rule](../05_Multivariable_Differential_Calculus/08_multivariable_chain_rule.ipynb). For the prototypical layer $\mathbf{r} = W\mathbf{x} - \mathbf{b}$ with loss $L = \tfrac12\|\mathbf{r}\|^2$, the chain rule gives $\partial L/\partial\mathbf{r} = \mathbf{r}$ and then $\partial L/\partial W = \mathbf{r}\,\mathbf{x}^\top$ — an outer product of the upstream error with the input. This is exactly the weight-gradient formula of a linear layer in [backpropagation](./08_reverse_mode_automatic_differentiation.ipynb), the single most evaluated derivative in machine learning.

**Properties:**
- The "upstream gradient times local Jacobian" pattern is what each layer applies in reverse-mode AD.
- The outer-product structure $\mathbf{r}\,\mathbf{x}^\top$ has the same shape as $W$, ready for a gradient step.

**Numerical Example:**

Let $W = \begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix}$, $\mathbf{x} = [1, 1]^\top$, $\mathbf{b} = [2, 1]^\top$. The residual is

$$
\mathbf{r} = W\mathbf{x} - \mathbf{b} = \begin{bmatrix} 3 \\ 1 \end{bmatrix} - \begin{bmatrix} 2 \\ 1 \end{bmatrix} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}.
$$

Then the weight gradient is the outer product

$$
\frac{\partial L}{\partial W} = \mathbf{r}\,\mathbf{x}^\top = \begin{bmatrix} 1 \\ 0 \end{bmatrix}\begin{bmatrix} 1 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 1 \\ 0 & 0 \end{bmatrix}.
$$

In [1]:
import sympy as sp

W_11, W_12, W_21, W_22 = sp.symbols("W_11 W_12 W_21 W_22")
W = sp.Matrix([[W_11, W_12], [W_21, W_22]])
x = sp.Matrix([1, 1])
b = sp.Matrix([2, 1])

residual = W * x - b
loss = sp.Rational(1, 2) * (residual.T * residual)[0]
gradient_matrix = W.applyfunc(lambda entry: sp.diff(loss, entry))

evaluation = {W_11: 1, W_12: 2, W_21: 0, W_22: 1}
gradient_value = gradient_matrix.subs(evaluation)
outer_product = residual.subs(evaluation) * x.T

print("∂L/∂W (symbolic) ="); sp.pprint(sp.simplify(gradient_matrix))
print("∂L/∂W at W,x,b   ="); sp.pprint(gradient_value)
print("equals r·xᵀ      :", gradient_value == outer_product)

∂L/∂W (symbolic) =
⎡W₁₁ + W₁₂ - 2  W₁₁ + W₁₂ - 2⎤
⎢                            ⎥
⎣W₂₁ + W₂₂ - 1  W₂₁ + W₂₂ - 1⎦
∂L/∂W at W,x,b   =
⎡1  1⎤
⎢    ⎥
⎣0  0⎦
equals r·xᵀ      : True


**Equivalent CasADi implementation:**

CasADi recovers the same outer-product weight gradient by differentiating the loss with respect to the vectorized weights.

In [2]:
import casadi as ca

W = ca.SX.sym("W", 2, 2)
x = ca.DM([1, 1])
b = ca.DM([2, 1])

residual = W @ x - b
loss = 0.5 * ca.dot(residual, residual)
gradient_matrix = ca.reshape(ca.gradient(loss, ca.vec(W)), 2, 2)
gradient_function = ca.Function("dLdW", [W], [gradient_matrix])

print("∂L/∂W at W=[[1,2],[0,1]] =", gradient_function(ca.DM([[1, 2], [0, 1]])))

∂L/∂W at W=[[1,2],[0,1]] = 
[[1, 1], 
 [0, 0]]


**References:**

[📗 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*, Appendix A.4.](https://web.stanford.edu/~boyd/cvxbook/)  
[📘 Bertsekas, D. P. (1999). *Nonlinear Programming*, Appendix A.5.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Derivatives of Trace, Determinant, and Log-Det](./04_derivatives_of_trace_determinant_and_log_det.ipynb) | [Next: Jacobian-Vector and Vector-Jacobian Products ➡️](./06_jacobian_vector_and_vector_jacobian_products.ipynb)